# Step 6: Format text changes for manual review
During classification, Gemini can change the input text. 

Changes are recorded in a log file for review.

This script reformats the differences in the log file to make manual comparison between 'text_in' and 'text_out' pairs easier.

In [ ]:
# Load Log File
log_file_path = '03_classify_lines_gemini.log'

try:
    with open(log_file_path, 'r', encoding='utf-8') as file:
        log_content = file.read()
    print(f"Log file '{log_file_path}' loaded successfully.")
    print("First 500 characters of the log content:")
    print(log_content[:500])
except FileNotFoundError:
    print(f"Error: The file '{log_file_path}' was not found.")
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

Extract pairs of 'text_in' and 'text_out' lines, along with the source, page, column identifier.

In [ ]:
import re

# Regex pattern to capture the identifier from the ERROR line, text_in, and text_out pairs.
# This pattern looks for an ERROR line with 'dropped for' and an identifier, then 'text_in: ', then 'text_out: '.
pattern = r"ERROR:.*?dropped for\s*(.*?)\s*\(.*?:\s*text_in:\s*(.*?)\ntext_out:\s*(.*?)(?:\n|$)"

# Find all matches in the log content
matches = re.findall(pattern, log_content, re.DOTALL)

# Store extracted pairs in a list of dictionaries, including the identifier
extracted_pairs = []
for identifier, text_in, text_out in matches:
    extracted_pairs.append({
        'identifier': identifier.strip(),
        'text_in': text_in.strip(),
        'text_out': text_out.strip()
    })

print(f"Total number of extracted 'identifier'/'text_in'/'text_out' pairs: {len(extracted_pairs)}")
print("First 5 extracted pairs:")
for i, pair in enumerate(extracted_pairs[:5]):
    print(f"Pair {i+1}:")
    print(f"  Identifier: {pair['identifier']}")
    print(f"  text_in:    {pair['text_in'][:100]}...") # Truncate for display
    print(f"  text_out:   {pair['text_out'][:100]}...") # Truncate for display


`compare_texts` highlights the differenes by structuring them according to change type, affected text, and 10 characters of surrounding context for each `text_in` and `text_out` pair. 

The main loop iterates over all the differences in the log file, using `compare_texts` to identify specific changes, then displaying them including padding to help visualize what's added or cut.

In [ ]:
import difflib

def compare_texts(text1, text2):
    differences = []
    matcher = difflib.SequenceMatcher(None, text1, text2)

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal':
            continue

        change_type = tag
        affected_text_in = text1[i1:i2]
        affected_text_out = text2[j1:j2]

        # Get 10 characters of surrounding context
        context_before_in = text1[max(0, i1 - 10):i1]
        context_after_in = text1[i2:min(len(text1), i2 + 10)]
        context_before_out = text2[max(0, j1 - 10):j1]
        context_after_out = text2[j2:min(len(text2), j2 + 10)]

        differences.append({
            'change_type': change_type,
            'affected_text_in': affected_text_in,
            'affected_text_out': affected_text_out,
            'context_before_in': context_before_in,
            'context_after_in': context_after_in,
            'context_before_out': context_before_out,
            'context_after_out': context_after_out
        })
    return differences

# Initialize a list to store pairs that have differences
all_differing_pairs = []

print("Analyzing differences between text_in and text_out pairs...")

total_diffs = 0
for pair_num, pair in enumerate(extracted_pairs):
    pair_differences = compare_texts(pair['text_in'], pair['text_out'])

    if pair_differences:
        all_differing_pairs.append({
            'identifier': pair['identifier'],
            'text_in': pair['text_in'],
            'text_out': pair['text_out'],
            'differences': pair_differences
        })

        print(f"\nDifferences for Identifier: {pair['identifier']} (Pair {pair_num + 1}):") # Using identifier for display
        for diff in pair_differences:
            # Calculate max length for padding
            len_in = len(diff['affected_text_in'])
            len_out = len(diff['affected_text_out'])
            max_len = max(len_in, len_out)

            # Pad the strings for consistent display
            padded_affected_text_in = f"{diff['affected_text_in']:<{max_len}}"
            padded_affected_text_out = f"{diff['affected_text_out']:<{max_len}}"

            print(f"  Change Type: {diff['change_type']}")
            # Highlight changes using square brackets and padded text
            print(f"  text_in:  '{diff['context_before_in']}{{{{{padded_affected_text_in}}}}}{diff['context_after_in']}'")
            print(f"  text_out: '{diff['context_before_out']}{{{{{padded_affected_text_out}}}}}{diff['context_after_out']}'")
            total_diffs+=1
        print("--------------------------------------------------")

print(f"\nTotal number of pairs with differences found: {len(all_differing_pairs)}")
print(f"Total differences found: {total_diffs}")

Iterate through the `all_differing_pairs` list to gather statistics on the total number of differing pairs and count occurrences of each change type (insert, delete, replace). 

Collect examples of each change type for the summary.


In [ ]:
total_differing_pairs = 0
insert_count = 0
delete_count = 0
replace_count = 0

insert_example = None
delete_example = None
replace_example = None

for pair_info in all_differing_pairs:
    total_differing_pairs += 1

    for diff in pair_info['differences']:
        change_type = diff['change_type']

        if change_type == 'insert':
            insert_count += 1
            if insert_example is None:
                insert_example = diff
        elif change_type == 'delete':
            delete_count += 1
            if delete_example is None:
                delete_example = diff
        elif change_type == 'replace':
            replace_count += 1
            if replace_example is None:
                replace_example = diff

total_individual_differences = insert_count + delete_count + replace_count

print(f"Summary of Differences:")
print(f"Total pairs with differences: {total_differing_pairs}")
print(f"Total 'insert' changes: {insert_count}")
print(f"Total 'delete' changes: {delete_count}")
print(f"Total 'replace' changes: {replace_count}")
print(f"Total individual differences: {total_individual_differences}")

print("\nExample 'insert' difference:")
if insert_example:
    print(f"  text_in:  '{insert_example['context_before_in']}{{{{{insert_example['affected_text_in']}}}}}{insert_example['context_after_in']}'")
    print(f"  text_out: '{insert_example['context_before_out']}{{{{{insert_example['affected_text_out']}}}}}{insert_example['context_after_out']}'")
else:
    print("  No insert differences found.")

print("\nExample 'delete' difference:")
if delete_example:
    print(f"  text_in:  '{delete_example['context_before_in']}{{{{{delete_example['affected_text_in']}}}}}{delete_example['context_after_in']}'")
    print(f"  text_out: '{delete_example['context_before_out']}{{{{{delete_example['affected_text_out']}}}}}{delete_example['context_after_out']}'")
else:
    print("  No delete differences found.")

print("\nExample 'replace' difference:")
if replace_example:
    print(f"  text_in:  '{replace_example['context_before_in']}{{{{{replace_example['affected_text_in']}}}}}{replace_example['context_after_in']}'")
    print(f"  text_out: '{replace_example['context_before_out']}{{{{{replace_example['affected_text_out']}}}}}{replace_example['context_after_out']}'")
else:
    print("  No replace differences found.")

Create a CSV file named `differences_summary.csv` containing a summary of all identified differences, with columns for `identifier`, `text_in_context`, and `text_out_context`. 

Prepare difference data for CSV.
For each individual difference, keep 10 characters before, the padded affected text enclosed in `{{}}`, and 10 characters after. 
Store these with the `identifier` in a list of dictionaries.


In [ ]:
csv_data = []

for pair_info in all_differing_pairs:
    identifier = pair_info['identifier']

    for diff in pair_info['differences']:
        # Determine max length for padding
        len_in = len(diff['affected_text_in'])
        len_out = len(diff['affected_text_out'])
        max_len = max(len_in, len_out)

        # Pad the strings for consistent display
        padded_affected_text_in = f"{diff['affected_text_in']:<{max_len}}"
        padded_affected_text_out = f"{diff['affected_text_out']:<{max_len}}"

        # Construct context strings
        text_in_context = f"{diff['context_before_in']}{{{padded_affected_text_in}}}{diff['context_after_in']}"
        text_out_context = f"{diff['context_before_out']}{{{padded_affected_text_out}}}{diff['context_after_out']}"

        csv_data.append({
            'identifier': identifier,
            'text_in_context': text_in_context,
            'text_out_context': text_out_context
        })

print(f"Total entries prepared for CSV: {len(csv_data)}")
print("First 5 entries in csv_data:")
for i, entry in enumerate(csv_data[:5]):
    print(f"Entry {i+1}:")
    print(f"  Identifier: {entry['identifier']}")
    print(f"  text_in_context:  '{entry['text_in_context']}'")
    print(f"  text_out_context: '{entry['text_out_context']}'")


Convert the prepared list of dictionaries (`csv_data`) into a DataFrame and then save it as a CSV file.



In [ ]:
import pandas as pd

# Convert the list of dictionaries to a pandas DataFrame
df_differences = pd.DataFrame(csv_data)

# Save the DataFrame to a CSV file
output_csv_path = 'differences_summary.csv'
df_differences.to_csv(output_csv_path, index=False, encoding='utf-8')

print(f"CSV file '{output_csv_path}' created successfully with {len(df_differences)} entries.")
print("First 5 rows of the generated CSV (as a DataFrame preview):")
print(df_differences.head())